# Train a Piper voice in your own voice

Fine-tunes Piper's `en_US-lessac-medium` voice on your clips, on Colab's free GPU.

**Before you start**
1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. In Google Drive, make a folder `piper_voice` and upload `voice_dataset.zip` into it
   (from `data/voice_clips/` on your PC, made by `scripts/prepare_voice_clips.py`).

Then run the cells in order. Training saves to Drive every 25 epochs. If Colab disconnects
(free sessions end after a few hours, or sooner when idle), reconnect and run all cells
again: it carries on from the last save. Keep the Drive folder private: it holds your voice.

In [ ]:
# 1. Connect Google Drive and check the GPU
from google.colab import drive
drive.mount("/content/drive")

import pathlib
import subprocess

WORK = pathlib.Path("/content/drive/MyDrive/piper_voice")
VOICE = "en_US-omer-medium"
DATASET_ZIP = WORK / "voice_dataset.zip"
LATEST = WORK / "latest.ckpt"  # training progress (about 850 MB), replaced at each save
BASE = "en/en_US/lessac/medium/epoch=2164-step=1355540.ckpt"  # in rhasspy/piper-checkpoints
BASE_EPOCH = 2164  # training counts on from the base voice's epochs
EXTRA_EPOCHS = 1000  # how long to fine-tune; raise it if step 5 doesn't sound like you yet
SAVE_EVERY = 25  # epochs

assert DATASET_ZIP.exists(), f"Upload voice_dataset.zip to {WORK} in Google Drive first"
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print(gpu or "No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again.")

In [ ]:
%%bash
# 2. Install Piper's trainer (about 5 minutes; needed again in every new session)
set -e
cd /content
[ -d piper1-gpl ] || git clone -q --depth 1 --branch v1.8.0 https://github.com/OHF-voice/piper1-gpl.git
cd piper1-gpl
apt-get -qq install -y build-essential cmake ninja-build > /dev/null
pip install -q -e '.[train]' 'scikit-build<1'
./build_monotonic_align.sh
python setup.py -q build_ext --inplace > /dev/null 2>&1
python -c "from piper.train.vits import monotonic_align; from piper import espeakbridge; print('Piper trainer ready')"

In [ ]:
# 3. Unpack your clips
import shutil
import zipfile

DATA = pathlib.Path("/content/dataset")
shutil.rmtree(DATA, ignore_errors=True)
zipfile.ZipFile(DATASET_ZIP).extractall(DATA)
rows = (DATA / "metadata.csv").read_text(encoding="utf-8").splitlines()
print(f"{len(rows)} clips, {len(list((DATA / 'wav').glob('*.wav')))} wav files")

In [ ]:
%%writefile /content/train_voice.py
"""Piper's trainer, keeping one checkpoint in Google Drive instead of up to 11 local ones.

    python train_voice.py fit ...                                   train (Piper's options)
    python train_voice.py export --checkpoint X --output-file Y     Piper's ONNX export
"""
import os
import runpy
import shutil
import sys

import torch

_torch_load = torch.load


def trusted_load(*args, **kwargs):
    # Piper checkpoints store file paths, which PyTorch's safe loader (the default since 2.6)
    # rejects. Only Piper's official base checkpoint and your own saves are loaded here.
    kwargs["weights_only"] = False
    return _torch_load(*args, **kwargs)


torch.load = trusted_load

from lightning.pytorch.callbacks import Callback  # noqa: E402
from piper.train.__main__ import VitsLightningCLI  # noqa: E402
from piper.train.vits.dataset import VitsDataModule  # noqa: E402
from piper.train.vits.lightning import VitsModel  # noqa: E402


class TrainCLI(VitsLightningCLI):
    def _parse_ckpt_path(self):
        """Newer Lightning copies a checkpoint's settings over the command line's. The base
        checkpoint comes from Piper's older trainer, whose settings don't fit this one, so the
        command line's are kept; the weights still load as usual."""


class SaveToDrive(Callback):
    def __init__(self, path: str, every: int):
        self.path, self.every = path, every

    def on_train_epoch_end(self, trainer, pl_module):
        if (trainer.current_epoch + 1) % self.every == 0:
            self.save(trainer)

    def on_train_end(self, trainer, pl_module):
        self.save(trainer)

    def save(self, trainer):
        trainer.save_checkpoint("/content/latest.ckpt")
        shutil.copyfile("/content/latest.ckpt", self.path + ".tmp")
        os.replace(self.path + ".tmp", self.path)  # a disconnect mid-copy leaves the last save whole
        print(f"epoch {trainer.current_epoch}: saved to Drive", flush=True)


if __name__ == "__main__":
    if sys.argv[1] == "export":
        sys.argv = ["export_onnx"] + sys.argv[2:]
        runpy.run_module("piper.train.export_onnx", run_name="__main__")
    else:
        save = SaveToDrive(os.environ["SAVE_PATH"], int(os.environ["SAVE_EVERY"]))
        TrainCLI(
            VitsModel,
            VitsDataModule,
            trainer_defaults={"max_epochs": -1, "callbacks": [save], "enable_checkpointing": False},
        )

In [ ]:
# 4. Train. Stop this cell (the square button) whenever you want to listen in step 5; up to
#    SAVE_EVERY epochs since the last save are lost. Run it again to carry on.
if LATEST.exists():
    start = LATEST
    print("Carrying on from your last save in Drive")
else:
    from huggingface_hub import hf_hub_download
    print("Downloading the base voice (850 MB, straight to Colab)")
    start = hf_hub_download("rhasspy/piper-checkpoints", BASE, repo_type="dataset", local_dir="/content/base")

!cd /content/piper1-gpl && SAVE_PATH="{LATEST}" SAVE_EVERY={SAVE_EVERY} PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True \
  python /content/train_voice.py fit \
    --data.voice_name {VOICE} \
    --data.csv_path /content/dataset/metadata.csv \
    --data.audio_dir /content/dataset/wav \
    --data.espeak_voice en-us \
    --data.cache_dir /content/cache \
    --data.config_path "{WORK}/{VOICE}.onnx.json" \
    --data.batch_size 16 \
    --model.sample_rate 22050 \
    --trainer.max_epochs {BASE_EPOCH + EXTRA_EPOCHS} \
    --trainer.default_root_dir /content/logs \
    --ckpt_path "{start}"

In [ ]:
%%writefile /content/say.py
"""Speak a sentence with an exported voice: say.py <model.onnx> <out.wav> <text>"""
import sys
import wave

from piper import PiperVoice

model, out, text = sys.argv[1:4]
chunks = list(PiperVoice.load(model).synthesize(text))
with wave.open(out, "wb") as wav_file:
    wav_file.setnchannels(1)
    wav_file.setsampwidth(chunks[0].sample_width)
    wav_file.setframerate(chunks[0].sample_rate)
    for chunk in chunks:
        wav_file.writeframes(chunk.audio_int16_bytes)

In [ ]:
# 5. Listen: export the last save as a Piper voice (into Drive) and read a sentence with it
SENTENCE = "Hello, this is my own voice, reading an answer from my documents."
onnx = WORK / f"{VOICE}.onnx"
pathlib.Path("/content/sample.wav").unlink(missing_ok=True)
if not LATEST.exists():
    print("Nothing to listen to yet: run step 4 until it prints 'saved to Drive' at least once.")
else:
    !cd /content/piper1-gpl && python /content/train_voice.py export --checkpoint "{LATEST}" --output-file "{onnx}"
    !cd /content/piper1-gpl && python /content/say.py "{onnx}" /content/sample.wav "{SENTENCE}"

    from IPython.display import Audio, display
    if pathlib.Path("/content/sample.wav").exists():
        display(Audio(filename="/content/sample.wav"))

## Use the voice in the app

When step 5 sounds like you, download these two files from the `piper_voice` folder in
Google Drive (about 65 MB in all) into `data/models/` in the project:

- `en_US-omer-medium.onnx`
- `en_US-omer-medium.onnx.json`

Then add `TTS_VOICE=en_US-omer-medium` to `.env` and restart the app.

You can delete `latest.ckpt` from Drive once you are happy with the voice; keep it if you may
want to train longer later.